# Predicción del IAI futuro

Este notebook predice el Índice de Idoneidad Agrícola (IAI) bajo escenarios de cambio climático, utilizando el sistema de modelos LightGBM por región entrenado previamente.

**Estructura de los datos futuros:**
- Dos años: 2030 y 2040 (columna `year`)
- Dos escenarios: SSP2-4.5 (sufijo `_245`) y SSP5-8.5 (sufijo `_585`)
- Variables climáticas por escenario: `tmax_XXX`, `tmin_XXX`, `ppt_XXX`, `vpdmean_XXX`
- Variables de suelo: sin sufijo (no cambian con el escenario)

**Proceso para cada combinación año-escenario:**
1. Filtrar por año y renombrar las variables del escenario a los nombres base del modelo
2. Asignar cada parcela a su región con el K-Means entrenado
3. Predecir el IAI con el modelo LightGBM de esa región
4. Repetir para cada cultivo

**Importante:** el IAI futuro se *predice* con el modelo, no se *calcula* con la fórmula. El modelo aprendió la relación entre variables edafoclimáticas e idoneidad, y esa relación es la que se aplica a las condiciones futuras.

## 1. Configuración e importación

In [1]:
import polars as pl
import numpy as np
import joblib
import json
import os

# --- Rutas ---
MODELS_DIR = "../models"
LGB_DIR = os.path.join(MODELS_DIR, "lightgbm_final")

# --- Variables (deben coincidir EXACTAMENTE con el entrenamiento) ---
vars_clima = ["tmax", "tmin", "ppt"]
vars_suelo = ["ph1to1h2o_r", "awc_r", "profundidad_efectiva_cm","claytotal_r",
              "dbthirdbar_r", "sandtotal_r", "silttotal_r"]

# Variables usadas por el K-Means (mismas que en el clustering)
vars_cluster = vars_clima + vars_suelo

# --- Cultivos finales (códigos CDL) ---
FINAL_CROPS = [75, 69, 204, 76, 54, 3, 221, 212, 36, 24, 227, 2]

# --- Escenarios y años ---
ESCENARIOS = {"245": "SSP2-4.5", "585": "SSP5-8.5"}
ANIOS = [2030, 2040]


## 2. Cargar los modelos entrenados

Se cargan el scaler y el K-Means del clustering, más los cinco modelos LightGBM finales.

In [2]:
# Scaler y K-Means del clustering
# Ajusta los nombres si los guardaste distinto
scaler_cluster = joblib.load(os.path.join(MODELS_DIR, "scaler_cluster.pkl"))
kmeans = joblib.load(os.path.join(MODELS_DIR, "kmeans_clusters.pkl"))

# Los 5 modelos LightGBM finales
modelos = {}
for c in range(5):
    modelo = joblib.load(os.path.join(LGB_DIR, f"modelo_cluster_{c}_lgb.pkl"))
    with open(os.path.join(LGB_DIR, f"features_cluster_{c}.json")) as f:
        feats = json.load(f)
    modelos[c] = {"modelo": modelo, "features": feats}

print("Modelos cargados:")
print(f"  K-Means: {kmeans.n_clusters} clusters")
print(f"  LightGBM: {len(modelos)} modelos por región")
# Verificar los nombres de las features one-hot de un cluster
print(f"\nEjemplo de features del cluster 0 (últimas 5):")
print(f"  {modelos[0]['features'][-5:]}")


Modelos cargados:
  K-Means: 5 clusters
  LightGBM: 5 modelos por región

Ejemplo de features del cluster 0 (últimas 5):
  ['crop_name_Rice', 'crop_name_Strawberries', 'crop_name_Tomatoes', 'crop_name_Walnuts', 'crop_name_Wheat']


## 3. Cargar los datos futuros

Se cargan las proyecciones desde el Parquet (o MongoDB). Cada fila es una parcela en un año concreto, con las variables de ambos escenarios como columnas.

In [3]:
# Ajusta a tu fuente de datos futuros
RUTA_FUTURO = "../Data/fut_clean.parquet"
df_fut = pl.read_parquet(RUTA_FUTURO)

print(f"Filas totales: {df_fut.height:,}")
print(f"Años disponibles: {sorted(df_fut['year'].unique().to_list())}")
print(f"\nColumnas: {df_fut.columns}")


Filas totales: 65,830
Años disponibles: [2030, 2040]

Columnas: ['year', 'crop_id', 'crop_name', 'is_target', 'location', 'processed', 'ppt', 'processed_climate', 'tmax', 'tmean', 'tmin', 'vpdmax', 'vpdmin', 'awc_r', 'cec7_r', 'claytotal_r', 'dbthirdbar_r', 'ec_r', 'ksat_r', 'om_r', 'ph1to1h2o_r', 'processed_soil', 'profundidad_efectiva_cm', 'sandtotal_r', 'silttotal_r', 'sumbases_r', 'es_proyeccion', 'hurs_245', 'hurs_585', 'pr_245', 'pr_585', 'referencia_base', 'tmax_245', 'tmax_585', 'tmean_245', 'tmean_585', 'tmin_245', 'tmin_585', 'vpdmean', 'lon', 'lat', 'vpdmean_245', 'vpdmean_585', 'county']


In [4]:
columnas_a_eliminar = ['tmax', 'tmean', 'tmin', 'vpdmax', 'vpdmin', 'ppt', 'vpdmean']
df_fut = df_fut.drop(columnas_a_eliminar)
print(f"\nColumnas: {df_fut.columns}")



Columnas: ['year', 'crop_id', 'crop_name', 'is_target', 'location', 'processed', 'processed_climate', 'awc_r', 'cec7_r', 'claytotal_r', 'dbthirdbar_r', 'ec_r', 'ksat_r', 'om_r', 'ph1to1h2o_r', 'processed_soil', 'profundidad_efectiva_cm', 'sandtotal_r', 'silttotal_r', 'sumbases_r', 'es_proyeccion', 'hurs_245', 'hurs_585', 'pr_245', 'pr_585', 'referencia_base', 'tmax_245', 'tmax_585', 'tmean_245', 'tmean_585', 'tmin_245', 'tmin_585', 'lon', 'lat', 'vpdmean_245', 'vpdmean_585', 'county']


## 4. Función de preparación por año y escenario

Para cada combinación, se seleccionan las variables del escenario y se renombran a los nombres base que espera el modelo. El suelo se mantiene igual.

In [5]:
def preparar_esc_anio(df, escenario, anio):
    """
    Filtra por año, renombra las variables del escenario a nombres base,
    y devuelve un dataframe listo para clustering y predicción.
    """
    # Filtrar por año
    d = df.filter(pl.col("year") == anio)

    # Renombrar variables del escenario a nombres base
    renombres = {
        f"tmax_{escenario}": "tmax",
        f"tmin_{escenario}": "tmin",
        f"pr_{escenario}": "ppt",
        f"vpdmean_{escenario}": "vpdmean",
    }
    # Solo renombrar las que existan
    renombres = {k: v for k, v in renombres.items() if k in d.columns}
    d = d.rename(renombres)

    # Verificar que estén todas las variables necesarias
    faltantes = [v for v in vars_cluster if v not in d.columns]
    if faltantes:
        raise ValueError(f"Faltan variables para {escenario}-{anio}: {faltantes}")

    return d


In [6]:
# Mapeo código CDL -> nombre EN INGLÉS (debe coincidir con las columnas del modelo)
CROP_DICT_EN = {
    75: "Almonds", 69: "Grapes", 204: "Pistachios", 76: "Walnuts",
    54: "Tomatoes", 3: "Rice", 221: "Strawberries", 212: "Oranges",
    36: "Alfalfa", 24: "Wheat", 227: "Lettuce", 2: "Cotton",
}

## 5. Función de predicción del IAI

Asigna cluster con K-Means y predice con el LightGBM correspondiente, para cada cultivo. Construye el one-hot del cultivo según los nombres exactos que espera cada modelo.

In [7]:
def predecir_iai(df_prep, escenario_nombre, anio):
    # 1. Asignar cluster
    X_cl = df_prep.select(vars_cluster).to_numpy()
    X_cl_sc = scaler_cluster.transform(X_cl)
    clusters = kmeans.predict(X_cl_sc)
    df_prep = df_prep.with_columns(pl.Series("cluster", clusters))

    resultados = []
    for crop in FINAL_CROPS:
        nombre_cultivo = CROP_DICT_EN[crop]          # <-- código a nombre
        col_onehot = f"crop_name_{nombre_cultivo}"   # <-- nombre de la columna esperada

        for c in range(5):
            sub = df_prep.filter(pl.col("cluster") == c)
            if sub.height == 0:
                continue

            feats = modelos[c]["features"]
            X = np.zeros((sub.height, len(feats)), dtype=np.float32)
            for j, f in enumerate(feats):
                if f in sub.columns:
                    X[:, j] = sub[f].to_numpy()
                elif f == col_onehot:            # <-- compara con el nombre correcto
                    X[:, j] = 1.0
                # las demás columnas crop_name_* quedan en 0

            pred = np.clip(modelos[c]["modelo"].predict(X), 0, 1)

            resultados.append(
                sub.select(["lon", "lat"]).with_columns([
                    pl.lit(crop).alias("crop_id"),
                    pl.lit(c).alias("cluster"),
                    pl.Series("iai_pred", pred),
                    pl.lit(escenario_nombre).alias("escenario"),
                    pl.lit(anio).alias("anio"),
                ])
            )
    return pl.concat(resultados)

## 6. Ejecutar para todas las combinaciones

Se recorren los dos escenarios y los dos años, generando las predicciones de todos los cultivos.

In [8]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning,
)

In [9]:
todas = []
for esc, esc_nombre in ESCENARIOS.items():
    for anio in ANIOS:
        print(f"Procesando {esc_nombre} - {anio}...")
        df_prep = preparar_esc_anio(df_fut, esc, anio)
        pred = predecir_iai(df_prep, esc_nombre, anio)
        todas.append(pred)
        print(f"  {pred.height:,} predicciones")

iai_futuro = pl.concat(todas)
print(f"\nTotal de predicciones futuras: {iai_futuro.height:,}")
iai_futuro.head()


Procesando SSP2-4.5 - 2030...
  394,980 predicciones
Procesando SSP2-4.5 - 2040...
  394,980 predicciones
Procesando SSP5-8.5 - 2030...
  394,980 predicciones
Procesando SSP5-8.5 - 2040...
  394,980 predicciones

Total de predicciones futuras: 1,579,920


lon,lat,crop_id,cluster,iai_pred,escenario,anio
f64,f64,i32,i32,f64,str,i32
-121.198475,38.127808,75,2,0.867303,"""SSP2-4.5""",2030
-120.53668,37.37257,75,2,0.705077,"""SSP2-4.5""",2030
-120.868291,37.553503,75,2,0.730955,"""SSP2-4.5""",2030
-120.010622,36.724198,75,2,0.70217,"""SSP2-4.5""",2030
-116.671677,33.669819,75,2,0.880767,"""SSP2-4.5""",2030


## 7. Guardar resultados

In [10]:
iai_futuro.write_parquet("../Data/iai_futuro_predicho.parquet")
print("Guardado: iai_futuro_predicho.parquet")

# Resumen: IAI medio por escenario, año y cultivo
resumen = (
    iai_futuro
    .group_by(["escenario", "anio", "crop_id"])
    .agg(pl.col("iai_pred").mean().round(3).alias("iai_medio"))
    .sort(["escenario", "anio", "crop_id"])
)
print(resumen)


Guardado: iai_futuro_predicho.parquet
shape: (48, 4)
┌───────────┬──────┬─────────┬───────────┐
│ escenario ┆ anio ┆ crop_id ┆ iai_medio │
│ ---       ┆ ---  ┆ ---     ┆ ---       │
│ str       ┆ i32  ┆ i32     ┆ f64       │
╞═══════════╪══════╪═════════╪═══════════╡
│ SSP2-4.5  ┆ 2030 ┆ 2       ┆ 0.718     │
│ SSP2-4.5  ┆ 2030 ┆ 3       ┆ 0.681     │
│ SSP2-4.5  ┆ 2030 ┆ 24      ┆ 0.621     │
│ SSP2-4.5  ┆ 2030 ┆ 36      ┆ 0.751     │
│ SSP2-4.5  ┆ 2030 ┆ 54      ┆ 0.754     │
│ …         ┆ …    ┆ …       ┆ …         │
│ SSP5-8.5  ┆ 2040 ┆ 76      ┆ 0.626     │
│ SSP5-8.5  ┆ 2040 ┆ 204     ┆ 0.743     │
│ SSP5-8.5  ┆ 2040 ┆ 212     ┆ 0.667     │
│ SSP5-8.5  ┆ 2040 ┆ 221     ┆ 0.721     │
│ SSP5-8.5  ┆ 2040 ┆ 227     ┆ 0.601     │
└───────────┴──────┴─────────┴───────────┘


In [11]:
# Verificar que cada cultivo predice distinto
check = (
    iai_futuro
    .filter((pl.col("escenario")=="SSP5-8.5") & (pl.col("anio")==2040))
    .group_by("crop_id")
    .agg(pl.col("iai_pred").mean().round(4).alias("iai_medio"))
    .sort("iai_medio")
)
print(check)

shape: (12, 2)
┌─────────┬───────────┐
│ crop_id ┆ iai_medio │
│ ---     ┆ ---       │
│ i32     ┆ f64       │
╞═════════╪═══════════╡
│ 24      ┆ 0.5356    │
│ 227     ┆ 0.6012    │
│ 76      ┆ 0.6264    │
│ 212     ┆ 0.6673    │
│ 3       ┆ 0.6675    │
│ …       ┆ …         │
│ 221     ┆ 0.7211    │
│ 75      ┆ 0.7226    │
│ 54      ┆ 0.7321    │
│ 36      ┆ 0.7348    │
│ 204     ┆ 0.7426    │
└─────────┴───────────┘


## 8. Análisis: comparación presente vs futuro

Para evaluar el efecto del cambio climático, se compara el IAI futuro con el histórico. También se identifican las parcelas que cambian de región, lo que indica un desplazamiento de los regímenes biofísicos.

In [12]:
df_hist = pl.read_parquet("../Data/hist_iai.parquet")

In [13]:
CROP_ES = {
    75: "Almendras", 69: "Uvas", 204: "Pistachos", 76: "Nueces",
    54: "Tomates", 3: "Arroz", 221: "Fresas", 212: "Naranjas",
    36: "Alfalfa", 24: "Trigo", 227: "Lechuga", 2: "Algodón",
}

comparacion = (
    df_hist.group_by("crop_id").agg(pl.col("IAI").mean().alias("iai_historico"))
    .join(
        iai_futuro.filter((pl.col("escenario")=="SSP5-8.5") & (pl.col("anio")==2040))
        .group_by("crop_id").agg(pl.col("iai_pred").mean().alias("iai_2040")),
        on="crop_id"
    )
    .with_columns([
        (pl.col("iai_2040") - pl.col("iai_historico")).round(4).alias("cambio"),
        pl.col("crop_id").replace_strict(CROP_ES).alias("cultivo"),
    ])
    .select(["cultivo", "iai_historico", "iai_2040", "cambio"])
    .sort("cambio")
)

with pl.Config(tbl_rows=20):
    print(comparacion)

shape: (12, 4)
┌───────────┬───────────────┬──────────┬─────────┐
│ cultivo   ┆ iai_historico ┆ iai_2040 ┆ cambio  │
│ ---       ┆ ---           ┆ ---      ┆ ---     │
│ str       ┆ f64           ┆ f64      ┆ f64     │
╞═══════════╪═══════════════╪══════════╪═════════╡
│ Trigo     ┆ 0.567987      ┆ 0.53563  ┆ -0.0324 │
│ Nueces    ┆ 0.654105      ┆ 0.626431 ┆ -0.0277 │
│ Almendras ┆ 0.659715      ┆ 0.72264  ┆ 0.0629  │
│ Fresas    ┆ 0.654221      ┆ 0.721126 ┆ 0.0669  │
│ Pistachos ┆ 0.665279      ┆ 0.742619 ┆ 0.0773  │
│ Arroz     ┆ 0.553218      ┆ 0.667454 ┆ 0.1142  │
│ Alfalfa   ┆ 0.600457      ┆ 0.73482  ┆ 0.1344  │
│ Uvas      ┆ 0.565206      ┆ 0.704863 ┆ 0.1397  │
│ Lechuga   ┆ 0.447956      ┆ 0.601244 ┆ 0.1533  │
│ Tomates   ┆ 0.572086      ┆ 0.732077 ┆ 0.16    │
│ Naranjas  ┆ 0.46345       ┆ 0.667317 ┆ 0.2039  │
│ Algodón   ┆ 0.428301      ┆ 0.705349 ┆ 0.277   │
└───────────┴───────────────┴──────────┴─────────┘


In [14]:
# IAI histórico por cultivo
hist = df_hist.group_by("crop_id").agg(pl.col("IAI").mean().alias("historico"))

# IAI futuro medio por cultivo, escenario y año
fut = (
    iai_futuro
    .group_by(["crop_id", "escenario", "anio"])
    .agg(pl.col("iai_pred").mean().alias("iai"))
)

# Pivotar: una columna por combinación escenario-año
fut_wide = fut.pivot(
    values="iai",
    index="crop_id",
    on=["escenario", "anio"],
)

# Unir histórico + futuro y poner nombres
tabla = (
    hist.join(fut_wide, on="crop_id")
    .with_columns(pl.col("crop_id").replace_strict(CROP_ES).alias("cultivo"))
    .sort("historico", descending=True)
)

# Reordenar columnas: cultivo, histórico, y las 4 combinaciones
cols_orden = ["cultivo", "historico"] + [c for c in tabla.columns
              if c not in ["cultivo", "historico", "crop_id"]]
tabla = tabla.select(cols_orden)

with pl.Config(tbl_rows=20, tbl_cols=10):
    print(tabla)

shape: (12, 6)
┌───────────┬───────────┬──────────────────┬──────────────────┬──────────────────┬─────────────────┐
│ cultivo   ┆ historico ┆ {"SSP2-4.5",2030 ┆ {"SSP5-8.5",2040 ┆ {"SSP5-8.5",2030 ┆ {"SSP2-4.5",204 │
│ ---       ┆ ---       ┆ }                ┆ }                ┆ }                ┆ 0}              │
│ str       ┆ f64       ┆ ---              ┆ ---              ┆ ---              ┆ ---             │
│           ┆           ┆ f64              ┆ f64              ┆ f64              ┆ f64             │
╞═══════════╪═══════════╪══════════════════╪══════════════════╪══════════════════╪═════════════════╡
│ Pistachos ┆ 0.665279  ┆ 0.766809         ┆ 0.742619         ┆ 0.784895         ┆ 0.836527        │
│ Almendras ┆ 0.659715  ┆ 0.753195         ┆ 0.72264          ┆ 0.769276         ┆ 0.845782        │
│ Fresas    ┆ 0.654221  ┆ 0.743278         ┆ 0.721126         ┆ 0.763681         ┆ 0.849991        │
│ Nueces    ┆ 0.654105  ┆ 0.655327         ┆ 0.626431         ┆ 0.658405    

In [15]:
fut = (
    iai_futuro
    .with_columns(
        (pl.col("escenario") + "_" + pl.col("anio").cast(pl.Utf8)).alias("esc_anio")
    )
    .group_by(["crop_id", "esc_anio"])
    .agg(pl.col("iai_pred").mean().alias("iai"))
)

fut_wide = fut.pivot(values="iai", index="crop_id", on="esc_anio")

tabla = (
    hist.join(fut_wide, on="crop_id")
    .with_columns(pl.col("crop_id").replace_strict(CROP_ES).alias("cultivo"))
    .sort("historico", descending=True)
    .select(["cultivo", "historico",
             "SSP2-4.5_2030", "SSP2-4.5_2040",
             "SSP5-8.5_2030", "SSP5-8.5_2040"])
    # Redondear todas las columnas numéricas con una expresión
    .with_columns(pl.col(pl.Float64).round(3))
)

with pl.Config(tbl_rows=20):
    print(tabla)

shape: (12, 6)
┌───────────┬───────────┬───────────────┬───────────────┬───────────────┬───────────────┐
│ cultivo   ┆ historico ┆ SSP2-4.5_2030 ┆ SSP2-4.5_2040 ┆ SSP5-8.5_2030 ┆ SSP5-8.5_2040 │
│ ---       ┆ ---       ┆ ---           ┆ ---           ┆ ---           ┆ ---           │
│ str       ┆ f64       ┆ f64           ┆ f64           ┆ f64           ┆ f64           │
╞═══════════╪═══════════╪═══════════════╪═══════════════╪═══════════════╪═══════════════╡
│ Pistachos ┆ 0.665     ┆ 0.767         ┆ 0.837         ┆ 0.785         ┆ 0.743         │
│ Almendras ┆ 0.66      ┆ 0.753         ┆ 0.846         ┆ 0.769         ┆ 0.723         │
│ Fresas    ┆ 0.654     ┆ 0.743         ┆ 0.85          ┆ 0.764         ┆ 0.721         │
│ Nueces    ┆ 0.654     ┆ 0.655         ┆ 0.759         ┆ 0.658         ┆ 0.626         │
│ Alfalfa   ┆ 0.6       ┆ 0.751         ┆ 0.848         ┆ 0.77          ┆ 0.735         │
│ Tomates   ┆ 0.572     ┆ 0.754         ┆ 0.847         ┆ 0.77          ┆ 0.732      

In [16]:
tabla = tabla.with_columns(
    (pl.col("SSP5-8.5_2040") - pl.col("historico")).round(4).alias("cambio_max")
)

In [17]:
# Ver TODAS las columnas de los datos futuros
print("Columnas de df_fut:")
for col in df_fut.columns:
    print(f"  {col}")

Columnas de df_fut:
  year
  crop_id
  crop_name
  is_target
  location
  processed
  processed_climate
  awc_r
  cec7_r
  claytotal_r
  dbthirdbar_r
  ec_r
  ksat_r
  om_r
  ph1to1h2o_r
  processed_soil
  profundidad_efectiva_cm
  sandtotal_r
  silttotal_r
  sumbases_r
  es_proyeccion
  hurs_245
  hurs_585
  pr_245
  pr_585
  referencia_base
  tmax_245
  tmax_585
  tmean_245
  tmean_585
  tmin_245
  tmin_585
  lon
  lat
  vpdmean_245
  vpdmean_585
  county


In [18]:
print("\nTemperatura media por escenario y año (df_fut):\n")
for esc in ["245", "585"]:
    for anio in [2030, 2040]:
        d = df_fut.filter(pl.col("year") == anio)
        tmax = d[f"tmax_{esc}"].mean()
        tmin = d[f"tmin_{esc}"].mean()
        tmax_s = f"{tmax:.3f}" if tmax is not None else "None"
        tmin_s = f"{tmin:.3f}" if tmin is not None else "None"
        print(f"  {esc} {anio}: tmax={tmax_s}, tmin={tmin_s}")

print("\nPrecipitación media por escenario y año (df_fut):\n")
for esc in ["245", "585"]:
    for anio in [2030, 2040]:
        d = df_fut.filter(pl.col("year") == anio)
        pr = d[f"pr_{esc}"].mean()
        pr_s = f"{pr:.1f}" if pr is not None else "None"
        print(f"  {esc} {anio}: pr={pr_s}")


Temperatura media por escenario y año (df_fut):

  245 2030: tmax=24.804, tmin=10.829
  245 2040: tmax=23.965, tmin=10.739
  585 2030: tmax=24.635, tmin=11.248
  585 2040: tmax=26.302, tmin=11.666

Precipitación media por escenario y año (df_fut):

  245 2030: pr=384.3
  245 2040: pr=705.6
  585 2030: pr=431.4
  585 2040: pr=356.7


In [19]:
print("Promedio de ambos años por escenario:\n")
for esc in ["245", "585"]:
    d = df_fut  # todos los años
    tmax = d[f"tmax_{esc}"].mean()
    tmin = d[f"tmin_{esc}"].mean()
    pr = d[f"pr_{esc}"].mean()
    print(f"  SSP{esc}: tmax={tmax:.2f}, tmin={tmin:.2f}, pr={pr:.1f}")

Promedio de ambos años por escenario:

  SSP245: tmax=24.38, tmin=10.78, pr=545.0
  SSP585: tmax=25.47, tmin=11.46, pr=394.0


##OPCION PROMEDIAR AÑOS

In [20]:
# Ajusta a tu fuente de datos futuros
RUTA_FUTURO = "../Data/fut_clean.parquet"
df_fut = pl.read_parquet(RUTA_FUTURO)

print(f"Filas totales: {df_fut.height:,}")
print(f"Años disponibles: {sorted(df_fut['year'].unique().to_list())}")
print(f"\nColumnas: {df_fut.columns}")


Filas totales: 65,830
Años disponibles: [2030, 2040]

Columnas: ['year', 'crop_id', 'crop_name', 'is_target', 'location', 'processed', 'ppt', 'processed_climate', 'tmax', 'tmean', 'tmin', 'vpdmax', 'vpdmin', 'awc_r', 'cec7_r', 'claytotal_r', 'dbthirdbar_r', 'ec_r', 'ksat_r', 'om_r', 'ph1to1h2o_r', 'processed_soil', 'profundidad_efectiva_cm', 'sandtotal_r', 'silttotal_r', 'sumbases_r', 'es_proyeccion', 'hurs_245', 'hurs_585', 'pr_245', 'pr_585', 'referencia_base', 'tmax_245', 'tmax_585', 'tmean_245', 'tmean_585', 'tmin_245', 'tmin_585', 'vpdmean', 'lon', 'lat', 'vpdmean_245', 'vpdmean_585', 'county']


In [21]:
columnas_a_eliminar = ['tmax', 'tmean', 'tmin', 'vpdmax', 'vpdmin', 'ppt', 'vpdmean']
df_fut = df_fut.drop(columnas_a_eliminar)
print(f"\nColumnas: {df_fut.columns}")


Columnas: ['year', 'crop_id', 'crop_name', 'is_target', 'location', 'processed', 'processed_climate', 'awc_r', 'cec7_r', 'claytotal_r', 'dbthirdbar_r', 'ec_r', 'ksat_r', 'om_r', 'ph1to1h2o_r', 'processed_soil', 'profundidad_efectiva_cm', 'sandtotal_r', 'silttotal_r', 'sumbases_r', 'es_proyeccion', 'hurs_245', 'hurs_585', 'pr_245', 'pr_585', 'referencia_base', 'tmax_245', 'tmax_585', 'tmean_245', 'tmean_585', 'tmin_245', 'tmin_585', 'lon', 'lat', 'vpdmean_245', 'vpdmean_585', 'county']


In [22]:
# Promediar las variables climáticas de 2030 y 2040 por parcela y escenario
# (el suelo no cambia, se toma igual)

vars_por_esc = ["tmax", "tmin", "pr", "vpdmean"]  # las que tienen sufijo

# Agrupar por parcela promediando ambos años
df_fut_avg = (
    df_fut
    .group_by(["lon", "lat"])
    .agg(
        # Promedio de las climáticas de ambos escenarios
        [pl.col(f"{v}_245").mean().alias(f"{v}_245") for v in vars_por_esc] +
        [pl.col(f"{v}_585").mean().alias(f"{v}_585") for v in vars_por_esc] +
        # El suelo es constante, tomamos el primero
        [pl.col(s).first().alias(s) for s in vars_suelo] +
        [pl.col("crop_name").first().alias("crop_name"),
         pl.col("crop_id").first().alias("crop_id")]
    )
)
print(f"Parcelas tras promediar años: {df_fut_avg.height:,}")

Parcelas tras promediar años: 32,915


In [23]:
def preparar_escenario(df, escenario):
    """Prepara un escenario ya promediado (sin filtrar año)."""
    renombres = {
        f"tmax_{escenario}": "tmax",
        f"tmin_{escenario}": "tmin",
        f"pr_{escenario}": "ppt",
        f"vpdmean_{escenario}": "vpdmean",
    }
    renombres = {k: v for k, v in renombres.items() if k in df.columns}
    d = df.rename(renombres)
    faltantes = [v for v in vars_cluster if v not in d.columns]
    if faltantes:
        raise ValueError(f"Faltan: {faltantes}")
    return d

# Predecir para cada escenario (sin año)
todas = []
for esc, esc_nombre in ESCENARIOS.items():
    df_prep = preparar_escenario(df_fut_avg, esc)
    pred = predecir_iai(df_prep, esc_nombre, "2030-2040")  # etiqueta de periodo
    todas.append(pred)

iai_futuro = pl.concat(todas)

In [24]:
def predecir_iai(df_prep, escenario_nombre, anio):
    # 1. Asignar cluster
    X_cl = df_prep.select(vars_cluster).to_numpy()
    X_cl_sc = scaler_cluster.transform(X_cl)
    clusters = kmeans.predict(X_cl_sc)
    df_prep = df_prep.with_columns(pl.Series("cluster", clusters))

    resultados = []
    for crop in FINAL_CROPS:
        nombre_cultivo = CROP_DICT_EN[crop]          # <-- código a nombre
        col_onehot = f"crop_name_{nombre_cultivo}"   # <-- nombre de la columna esperada

        for c in range(5):
            sub = df_prep.filter(pl.col("cluster") == c)
            if sub.height == 0:
                continue

            feats = modelos[c]["features"]
            X = np.zeros((sub.height, len(feats)), dtype=np.float32)
            for j, f in enumerate(feats):
                if f in sub.columns:
                    X[:, j] = sub[f].to_numpy()
                elif f == col_onehot:            # <-- compara con el nombre correcto
                    X[:, j] = 1.0
                # las demás columnas crop_name_* quedan en 0

            pred = np.clip(modelos[c]["modelo"].predict(X), 0, 1)

            resultados.append(
                sub.select(["lon", "lat"]).with_columns([
                    pl.lit(crop).alias("crop_id"),
                    pl.lit(c).alias("cluster"),
                    pl.Series("iai_pred", pred),
                    pl.lit(escenario_nombre).alias("escenario"),
                    pl.lit(anio).alias("anio"),
                ])
            )
    return pl.concat(resultados)

In [25]:
# Mapeo código CDL -> nombre EN INGLÉS (debe coincidir con las columnas del modelo)
CROP_DICT_EN = {
    75: "Almonds", 69: "Grapes", 204: "Pistachios", 76: "Walnuts",
    54: "Tomatoes", 3: "Rice", 221: "Strawberries", 212: "Oranges",
    36: "Alfalfa", 24: "Wheat", 227: "Lettuce", 2: "Cotton",
}

In [26]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning,
)

In [27]:
todas = []
for esc, esc_nombre in ESCENARIOS.items():
    for anio in ANIOS:
        print(f"Procesando {esc_nombre} - {anio}...")
        df_prep = preparar_esc_anio(df_fut, esc, anio)
        pred = predecir_iai(df_prep, esc_nombre, anio)
        todas.append(pred)
        print(f"  {pred.height:,} predicciones")

iai_futuro = pl.concat(todas)
print(f"\nTotal de predicciones futuras: {iai_futuro.height:,}")
iai_futuro.head()


Procesando SSP2-4.5 - 2030...
  394,980 predicciones
Procesando SSP2-4.5 - 2040...
  394,980 predicciones
Procesando SSP5-8.5 - 2030...
  394,980 predicciones
Procesando SSP5-8.5 - 2040...
  394,980 predicciones

Total de predicciones futuras: 1,579,920


lon,lat,crop_id,cluster,iai_pred,escenario,anio
f64,f64,i32,i32,f64,str,i32
-121.198475,38.127808,75,2,0.867303,"""SSP2-4.5""",2030
-120.53668,37.37257,75,2,0.705077,"""SSP2-4.5""",2030
-120.868291,37.553503,75,2,0.730955,"""SSP2-4.5""",2030
-120.010622,36.724198,75,2,0.70217,"""SSP2-4.5""",2030
-116.671677,33.669819,75,2,0.880767,"""SSP2-4.5""",2030


In [28]:
hist = df_hist.group_by("crop_id").agg(pl.col("IAI").mean().alias("historico"))
fut = (
    iai_futuro.group_by(["crop_id", "escenario"])
    .agg(pl.col("iai_pred").mean().alias("iai"))
)
fut_wide = fut.pivot(values="iai", index="crop_id", on="escenario")

tabla = (
    hist.join(fut_wide, on="crop_id")
    .with_columns(pl.col("crop_id").replace_strict(CROP_ES).alias("cultivo"))
    .sort("historico", descending=True)
    .with_columns(pl.col(pl.Float64).round(3))
)
with pl.Config(tbl_rows=20):
    print(tabla.select(["cultivo", "historico", "SSP2-4.5", "SSP5-8.5"]))

shape: (12, 4)
┌───────────┬───────────┬──────────┬──────────┐
│ cultivo   ┆ historico ┆ SSP2-4.5 ┆ SSP5-8.5 │
│ ---       ┆ ---       ┆ ---      ┆ ---      │
│ str       ┆ f64       ┆ f64      ┆ f64      │
╞═══════════╪═══════════╪══════════╪══════════╡
│ Pistachos ┆ 0.665     ┆ 0.802    ┆ 0.764    │
│ Almendras ┆ 0.66      ┆ 0.799    ┆ 0.746    │
│ Fresas    ┆ 0.654     ┆ 0.797    ┆ 0.742    │
│ Nueces    ┆ 0.654     ┆ 0.707    ┆ 0.642    │
│ Alfalfa   ┆ 0.6       ┆ 0.799    ┆ 0.752    │
│ Tomates   ┆ 0.572     ┆ 0.8      ┆ 0.751    │
│ Trigo     ┆ 0.568     ┆ 0.691    ┆ 0.593    │
│ Uvas      ┆ 0.565     ┆ 0.768    ┆ 0.721    │
│ Arroz     ┆ 0.553     ┆ 0.685    ┆ 0.672    │
│ Naranjas  ┆ 0.463     ┆ 0.704    ┆ 0.675    │
│ Lechuga   ┆ 0.448     ┆ 0.687    ┆ 0.625    │
│ Algodón   ┆ 0.428     ┆ 0.749    ┆ 0.716    │
└───────────┴───────────┴──────────┴──────────┘


In [29]:
# ══════════════════════════════════════════════════════
#  PREDECIR EL IAI DE 2025 CON EL MODELO
#  (para comparar con el IAI calculado y detectar sesgo)
# ══════════════════════════════════════════════════════

# 1. Tomar los datos de 2025 del histórico
df_2025 = df_hist.filter(pl.col("year") == 2025)
print(f"Parcelas en 2025: {df_2025.height:,}")

# Verificar que tenga las variables necesarias con nombre base
faltantes = [v for v in vars_cluster if v not in df_2025.columns]
if faltantes:
    print(f"ATENCIÓN, faltan columnas: {faltantes}")
else:
    print("Todas las variables del modelo están presentes.")

Parcelas en 2025: 32,915
Todas las variables del modelo están presentes.


In [30]:
def predecir_iai_presente(df_base, etiqueta="2025_predicho"):
    """
    Predice el IAI con el modelo sobre datos que YA tienen los nombres base
    (sin sufijo de escenario). Mismo proceso que el futuro: cluster + LightGBM.
    """
    # 1. Asignar cluster
    X_cl = df_base.select(vars_cluster).to_numpy()
    X_cl_sc = scaler_cluster.transform(X_cl)
    clusters = kmeans.predict(X_cl_sc)
    df_base = df_base.with_columns(pl.Series("cluster", clusters))

    resultados = []
    for crop in FINAL_CROPS:
        nombre_cultivo = CROP_DICT_EN[crop]
        col_onehot = f"crop_name_{nombre_cultivo}"

        for c in range(5):
            sub = df_base.filter(pl.col("cluster") == c)
            if sub.height == 0:
                continue

            feats = modelos[c]["features"]
            X = np.zeros((sub.height, len(feats)), dtype=np.float32)
            for j, f in enumerate(feats):
                if f in sub.columns:
                    X[:, j] = sub[f].to_numpy()
                elif f == col_onehot:
                    X[:, j] = 1.0

            pred = np.clip(modelos[c]["modelo"].predict(X), 0, 1)

            resultados.append(
                sub.select(["lon", "lat"]).with_columns([
                    pl.lit(crop).alias("crop_id"),
                    pl.lit(c).alias("cluster"),
                    pl.Series("iai_pred", pred),
                    pl.lit(etiqueta).alias("origen"),
                ])
            )
    return pl.concat(resultados)


# 2. Predecir 2025 con el modelo
iai_2025_pred = predecir_iai_presente(df_2025)
print(f"Predicciones 2025: {iai_2025_pred.height:,}")

Predicciones 2025: 394,980


In [31]:
# ══════════════════════════════════════════════════════
#  COMPARACIÓN CLAVE: 2025 calculado (fórmula) vs 2025 predicho (modelo)
#  Si difieren sistemáticamente, el modelo tiene sesgo
# ══════════════════════════════════════════════════════

# IAI 2025 CALCULADO (el que ya tienes, de la fórmula)
iai_2025_calc = (
    df_2025
    .group_by("crop_id")
    .agg(pl.col("IAI").mean().alias("iai_calculado"))
)

# IAI 2025 PREDICHO (el que acabas de generar con el modelo)
iai_2025_modelo = (
    iai_2025_pred
    .group_by("crop_id")
    .agg(pl.col("iai_pred").mean().alias("iai_predicho"))
)

# Comparar
comparacion_sesgo = (
    iai_2025_calc.join(iai_2025_modelo, on="crop_id")
    .with_columns([
        (pl.col("iai_predicho") - pl.col("iai_calculado")).alias("sesgo"),
        pl.col("crop_id").replace_strict(CROP_ES).alias("cultivo"),
    ])
    .select(["cultivo", "iai_calculado", "iai_predicho", "sesgo"])
    .sort("sesgo", descending=True)
)

with pl.Config(tbl_rows=20):
    print(comparacion_sesgo.with_columns(pl.col(pl.Float64).round(4)))

# Resumen del sesgo
sesgo_medio = comparacion_sesgo["sesgo"].mean()
print(f"\nSesgo medio del modelo (predicho - calculado): {sesgo_medio:+.4f}")
print("Si es positivo, el modelo INFLA los valores.")
print("Si es cercano a 0, el modelo no tiene sesgo sistemático.")

shape: (12, 4)
┌───────────┬───────────────┬──────────────┬────────┐
│ cultivo   ┆ iai_calculado ┆ iai_predicho ┆ sesgo  │
│ ---       ┆ ---           ┆ ---          ┆ ---    │
│ str       ┆ f64           ┆ f64          ┆ f64    │
╞═══════════╪═══════════════╪══════════════╪════════╡
│ Algodón   ┆ 0.4323        ┆ 0.6958       ┆ 0.2635 │
│ Naranjas  ┆ 0.4622        ┆ 0.6683       ┆ 0.2062 │
│ Lechuga   ┆ 0.4363        ┆ 0.6168       ┆ 0.1806 │
│ Tomates   ┆ 0.5523        ┆ 0.7235       ┆ 0.1712 │
│ Arroz     ┆ 0.5538        ┆ 0.6873       ┆ 0.1336 │
│ Uvas      ┆ 0.5728        ┆ 0.7046       ┆ 0.1318 │
│ Alfalfa   ┆ 0.5914        ┆ 0.7201       ┆ 0.1287 │
│ Pistachos ┆ 0.6692        ┆ 0.7657       ┆ 0.0965 │
│ Almendras ┆ 0.6621        ┆ 0.736        ┆ 0.0739 │
│ Trigo     ┆ 0.5981        ┆ 0.6333       ┆ 0.0352 │
│ Nueces    ┆ 0.6439        ┆ 0.6621       ┆ 0.0182 │
│ Fresas    ┆ 0.7153        ┆ 0.7254       ┆ 0.0102 │
└───────────┴───────────────┴──────────────┴────────┘

Sesgo medio 

In [32]:
# Comparación LIMPIA: 2025 predicho vs futuro predicho (mismo modelo, sesgo cancelado)
base_pred = (
    iai_2025_pred.group_by("crop_id")
    .agg(pl.col("iai_pred").mean().alias("iai_2025"))
)

fut_pred = (
    iai_futuro.group_by(["crop_id", "escenario"])
    .agg(pl.col("iai_pred").mean().alias("iai"))
)
fut_wide = fut_pred.pivot(values="iai", index="crop_id", on="escenario")

comparacion_limpia = (
    base_pred.join(fut_wide, on="crop_id")
    .with_columns([
        (pl.col("SSP2-4.5") - pl.col("iai_2025")).alias("cambio_245"),
        (pl.col("SSP5-8.5") - pl.col("iai_2025")).alias("cambio_585"),
        pl.col("crop_id").replace_strict(CROP_ES).alias("cultivo"),
    ])
    .select(["cultivo", "iai_2025", "SSP2-4.5", "SSP5-8.5", "cambio_245", "cambio_585"])
    .sort("cambio_585")
)

with pl.Config(tbl_rows=20):
    print(comparacion_limpia.with_columns(pl.col(pl.Float64).round(4)))

shape: (12, 6)
┌───────────┬──────────┬──────────┬──────────┬────────────┬────────────┐
│ cultivo   ┆ iai_2025 ┆ SSP2-4.5 ┆ SSP5-8.5 ┆ cambio_245 ┆ cambio_585 │
│ ---       ┆ ---      ┆ ---      ┆ ---      ┆ ---        ┆ ---        │
│ str       ┆ f64      ┆ f64      ┆ f64      ┆ f64        ┆ f64        │
╞═══════════╪══════════╪══════════╪══════════╪════════════╪════════════╡
│ Trigo     ┆ 0.6333   ┆ 0.691    ┆ 0.5925   ┆ 0.0577     ┆ -0.0408    │
│ Nueces    ┆ 0.6621   ┆ 0.7073   ┆ 0.6424   ┆ 0.0452     ┆ -0.0197    │
│ Arroz     ┆ 0.6873   ┆ 0.685    ┆ 0.6724   ┆ -0.0023    ┆ -0.015     │
│ Pistachos ┆ 0.7657   ┆ 0.8017   ┆ 0.7638   ┆ 0.0359     ┆ -0.002     │
│ Naranjas  ┆ 0.6683   ┆ 0.7035   ┆ 0.6754   ┆ 0.0352     ┆ 0.0071     │
│ Lechuga   ┆ 0.6168   ┆ 0.6869   ┆ 0.6252   ┆ 0.07       ┆ 0.0084     │
│ Almendras ┆ 0.736    ┆ 0.7995   ┆ 0.746    ┆ 0.0635     ┆ 0.01       │
│ Uvas      ┆ 0.7046   ┆ 0.7676   ┆ 0.7212   ┆ 0.063      ┆ 0.0166     │
│ Fresas    ┆ 0.7254   ┆ 0.7966   ┆ 

In [33]:
# Recuperar profundidad_efectiva_cm del histórico (no cambia en el futuro)
prof_hist = (
    df_hist.filter(pl.col("profundidad_efectiva_cm").is_not_null())
    .select(["lon", "lat", "profundidad_efectiva_cm"])
    .unique(subset=["lon", "lat"])
)

# Unir al df futuro
df_fut_avg = df_fut_avg.join(prof_hist, on=["lon", "lat"], how="left")

# Verificar cuántas parcelas quedaron con profundidad
n_sin = df_fut_avg.filter(pl.col("profundidad_efectiva_cm").is_null()).height
print(f"Parcelas sin profundidad: {n_sin}")

Parcelas sin profundidad: 0


In [34]:
print("¿profundidad_efectiva_cm en df_hist?:", "profundidad_efectiva_cm" in df_hist.columns)
# Si no está, busca cómo se llama:
print([c for c in df_hist.columns if "profund" in c.lower() or "depth" in c.lower()])

¿profundidad_efectiva_cm en df_hist?: True
['profundidad_efectiva_cm']


In [35]:
import sys
sys.path.append("../src")  # ajusta a donde esté IAI_estimador.py
from IAI_estimador import calculate_iai   # tu función

# ══════════════════════════════════════════════════════
#  IAI FUTURO CON LA FÓRMULA (no con el modelo)
# ══════════════════════════════════════════════════════

# La función necesita: tmax, tmin, ppt, ph1to1h2o_r,
# profundidad_efectiva_cm, awc_r, y crop_id
# Verifica que df_fut_avg tenga todas:
necesarias = ["tmax", "tmin", "ppt", "ph1to1h2o_r",
              "profundidad_efectiva_cm", "awc_r"]

def iai_formula_por_escenario(df_esc, escenario_nombre):
    """Aplica la fórmula del IAI a un escenario, para los 12 cultivos."""
    resultados = []
    for crop in FINAL_CROPS:
        # Asignar el crop_id y calcular el IAI con la fórmula
        df_crop = df_esc.with_columns(pl.lit(crop).alias("crop_id"))
        df_iai = calculate_iai(df_crop)   # tu función devuelve df con columna IAI
        resultados.append(
            df_iai.select(["lon", "lat", "crop_id", "IAI"])
            .with_columns(pl.lit(escenario_nombre).alias("escenario"))
        )
    return pl.concat(resultados)

# Preparar cada escenario (renombrar pr_XXX -> ppt, etc.)
iai_formula_todos = []
for esc, esc_nombre in ESCENARIOS.items():
    df_prep = preparar_escenario(df_fut_avg, esc)  # la función que renombra
    iai_f = iai_formula_por_escenario(df_prep, esc_nombre)
    iai_formula_todos.append(iai_f)

iai_formula_fut = pl.concat(iai_formula_todos)
print(f"IAI futuro por fórmula: {iai_formula_fut.height:,} filas")

IAI futuro por fórmula: 789,960 filas


In [36]:
# ══════════════════════════════════════════════════════
#  COMPARAR: fórmula histórica vs fórmula futura
#  (ambas con la MISMA fórmula, comparación homogénea)
# ══════════════════════════════════════════════════════

# IAI histórico calculado (fórmula) - el que ya tienes
hist_formula = df_hist.group_by("crop_id").agg(pl.col("IAI").mean().alias("hist"))

# IAI futuro por fórmula
fut_formula = (
    iai_formula_fut.group_by(["crop_id", "escenario"])
    .agg(pl.col("IAI").mean().alias("iai"))
)
fut_formula_wide = fut_formula.pivot(values="iai", index="crop_id", on="escenario")

tabla_formula = (
    hist_formula.join(fut_formula_wide, on="crop_id")
    .with_columns([
        (pl.col("SSP2-4.5") - pl.col("hist")).alias("cambio_245"),
        (pl.col("SSP5-8.5") - pl.col("hist")).alias("cambio_585"),
        pl.col("crop_id").replace_strict(CROP_ES).alias("cultivo"),
    ])
    .select(["cultivo", "hist", "SSP2-4.5", "SSP5-8.5", "cambio_245", "cambio_585"])
    .sort("cambio_585")
)

with pl.Config(tbl_rows=20):
    print(tabla_formula.with_columns(pl.col(pl.Float64).round(4)))

shape: (12, 6)
┌───────────┬────────┬──────────┬──────────┬────────────┬────────────┐
│ cultivo   ┆ hist   ┆ SSP2-4.5 ┆ SSP5-8.5 ┆ cambio_245 ┆ cambio_585 │
│ ---       ┆ ---    ┆ ---      ┆ ---      ┆ ---        ┆ ---        │
│ str       ┆ f64    ┆ f64      ┆ f64      ┆ f64        ┆ f64        │
╞═══════════╪════════╪══════════╪══════════╪════════════╪════════════╡
│ Fresas    ┆ 0.6542 ┆ 0.7287   ┆ 0.6503   ┆ 0.0744     ┆ -0.0039    │
│ Nueces    ┆ 0.6541 ┆ 0.6867   ┆ 0.6503   ┆ 0.0326     ┆ -0.0038    │
│ Arroz     ┆ 0.5532 ┆ 0.5595   ┆ 0.5732   ┆ 0.0063     ┆ 0.02       │
│ Naranjas  ┆ 0.4634 ┆ 0.5017   ┆ 0.4878   ┆ 0.0383     ┆ 0.0244     │
│ Trigo     ┆ 0.568  ┆ 0.6975   ┆ 0.5943   ┆ 0.1295     ┆ 0.0263     │
│ Uvas      ┆ 0.5652 ┆ 0.6406   ┆ 0.6199   ┆ 0.0754     ┆ 0.0547     │
│ Algodón   ┆ 0.4283 ┆ 0.5172   ┆ 0.4864   ┆ 0.0889     ┆ 0.0581     │
│ Almendras ┆ 0.6597 ┆ 0.7504   ┆ 0.7294   ┆ 0.0907     ┆ 0.0697     │
│ Tomates   ┆ 0.5721 ┆ 0.6734   ┆ 0.6437   ┆ 0.1013     ┆ 0.07

In [37]:
# Comparación climática: 2025 (presente) vs futuro por escenario y año
print("="*60)
print("  EVOLUCIÓN CLIMÁTICA: presente vs futuro")
print("="*60)

# Presente (2025) - variables base sin sufijo
d2025 = df_hist.filter(pl.col("year") == 2025)
print(f"\n2025 (PRESENTE, PRISM):")
print(f"  tmax = {d2025['tmax'].mean():.2f}°C")
print(f"  tmin = {d2025['tmin'].mean():.2f}°C")
print(f"  ppt  = {d2025['ppt'].mean():.1f} mm")

# Futuro por escenario y año
for esc, nombre in [("245", "SSP2-4.5 (moderado)"), ("585", "SSP5-8.5 (severo)")]:
    print(f"\n{nombre}:")
    for anio in [2030, 2040]:
        d = df_fut.filter(pl.col("year") == anio)
        tmax = d[f"tmax_{esc}"].mean()
        tmin = d[f"tmin_{esc}"].mean()
        pr = d[f"pr_{esc}"].mean()
        print(f"  {anio}: tmax={tmax:.2f}°C, tmin={tmin:.2f}°C, pr={pr:.1f} mm")

  EVOLUCIÓN CLIMÁTICA: presente vs futuro

2025 (PRESENTE, PRISM):
  tmax = 24.19°C
  tmin = 9.98°C
  ppt  = 329.6 mm

SSP2-4.5 (moderado):
  2030: tmax=24.80°C, tmin=10.83°C, pr=384.3 mm
  2040: tmax=23.97°C, tmin=10.74°C, pr=705.6 mm

SSP5-8.5 (severo):
  2030: tmax=24.64°C, tmin=11.25°C, pr=431.4 mm
  2040: tmax=26.30°C, tmin=11.67°C, pr=356.7 mm


In [38]:
df_hist_clean = pl.read_parquet("../Data/hist_clean.parquet")
print("Distribución de tmin anual en el histórico:")
print(f"  min:    {df_hist_clean['tmin'].min():.1f}")
print(f"  p10:    {df_hist_clean['tmin'].quantile(0.10):.1f}")
print(f"  media:  {df_hist_clean['tmin'].mean():.1f}")
print(f"  p90:    {df_hist_clean['tmin'].quantile(0.90):.1f}")
print(f"  max:    {df_hist_clean['tmin'].max():.1f}")

Distribución de tmin anual en el histórico:
  min:    -6.2
  p10:    7.7
  media:  9.4
  p90:    11.4
  max:    20.0


In [39]:
dif = df_hist_clean.with_columns(
    (pl.col("tmax") - (pl.col("tmax") + pl.col("tmin"))/2).alias("dif")
)
print(f"Diferencia media tmax - tmedia: {dif['dif'].mean():.2f}°C")

Diferencia media tmax - tmedia: 7.60°C
